# Week 1 — Day 2 (20 min): Train/Test Split (and why it matters)

**Goal (20 minutes):** Learn how to split data correctly, avoid common mistakes (like data leakage), and evaluate a model on unseen data.

You’ll:
- Create a **train/test split**
- See what happens when you **don’t split**
- Use **stratify** for classification
- Compare results across different **random states**


In [ ]:
# (Optional) If you're in a fresh Colab environment, uncomment:
# !pip -q install scikit-learn matplotlib


## 0) Quick warm-up (1 min)

In one sentence (comment):  
Why do we *not* train and test on the exact same data?


In [ ]:
# Your answer:
# Because ...


## 1) Load a dataset (3 min)

We’ll use the **Breast Cancer Wisconsin** dataset (binary classification).
- `X`: medical features
- `y`: 0/1 label (benign/malignant)


In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
X = data.data
y = data.target

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Class names:", data.target_names)
print("Class counts:", {0: (y==0).sum(), 1: (y==1).sum()})


## 2) The wrong way (on purpose): Train and test on the same data (4 min)

This often gives **overly optimistic** results because the model “sees” the answers during training.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

bad_model = LogisticRegression(max_iter=10000)
bad_model.fit(X, y)

bad_pred = bad_model.predict(X)
bad_acc = accuracy_score(y, bad_pred)

print("Accuracy when testing on TRAINING data (misleading):", bad_acc)


## 3) The correct way: Train/Test Split (6 min)

Key ideas:
- Train on one part, test on a **different** part.
- Use `random_state` so results are reproducible.
- Use `stratify=y` so the class ratio stays similar in both splits.


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Train size:", X_train.shape[0])
print("Test size:", X_test.shape[0])
print("Train class counts:", {0: (y_train==0).sum(), 1: (y_train==1).sum()})
print("Test class counts:", {0: (y_test==0).sum(), 1: (y_test==1).sum()})


### Train + evaluate (2 min)


In [ ]:
good_model = LogisticRegression(max_iter=10000)
good_model.fit(X_train, y_train)

y_pred = good_model.predict(X_test)
good_acc = accuracy_score(y_test, y_pred)

print("Accuracy on TEST data (realistic):", good_acc)


## 4) Mini experiment: random_state changes the split (3–6 min)

Run this cell a few times with different `seed` values.  
Notice how accuracy changes slightly because the train/test split changes.


In [ ]:
from sklearn.pipeline import make_pipeline

def eval_with_seed(seed: int) -> float:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=seed, stratify=y
    )
    pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=10000))
    pipe.fit(X_train, y_train)
    return accuracy_score(y_test, pipe.predict(X_test))

# Try different seeds here:
seeds = [0, 1, 2, 42, 99]

for s in seeds:
    print(s, "->", eval_with_seed(s))
